# AlBERTo enriched hashtag/PER/ORG inputs and preprocessing

# Enconder-only model pipeline

# Workflow

1. Load and inspect the data.
2. Select the political training and test subsets; inspect labels, duplicates, text lengths and special characters.
3. Clean and split the data. Remove URLs and mentions, separate emojis and normalize whitespace. Inspect conflicting labels before removing duplicate cleaned texts. The preprocessing also includes a hastag special treatment for extracting entities in the following phase.
4. Train - Val split 90% training / 10% validation split using seed 42, and check overlap between splits.

5. Prepare and fine-tune AlBERTo.
- Load the tokenizer and binary classification model.
- Count emojis only in the training partition, add frequent unsupported emojis and resize the embeddings.
- From a recent italian politicians and political parties name database extracted from WIKI data through the SPARQL endpoint find matchs in the tweets texts and add the tags <entity> word </entity>  around only match word when there is a hashtag such as #salvinimeloni or #salvini. This also includes a pre-processing of hastags.
- Tokenize cleaned text with a maximum length of 256 and dynamic padding.
- Fine-tune at 5e-6, evaluating every 100 steps and selecting the best validation macro-F1 checkpoint.


6. Evaluate and errors analysis
- Evaluation on the test set: accuracy, macro-F1, HATE precision/recall/F1 and the confusion matrix.
- Export validation/test predictions, false positives and false negatives.

Libraries import

In [ ]:
pip install emoji

In [ ]:
import pandas as pd
import re
import collections
import gc
from collections import Counter
import unicodedata

In [ ]:
import html
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)
from transformers import AddedToken
from transformers import EarlyStoppingCallback


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Import data

In [ ]:
training_textual_path = '/content/drive/MyDrive/master_unipa/training_textual.csv'
training_contextual_path = '/content/drive/MyDrive/master_unipa/training_contextual.csv'

test_textual_path = '/content/drive/MyDrive/master_unipa/test_textual_gold.csv'
test_contextual = '/content/drive/MyDrive/master_unipa/test_contextual.csv'

In [ ]:
df_train = pd.read_csv(training_textual_path)
df_train_contextual = pd.read_csv(training_contextual_path)

df_test = pd.read_csv(test_textual_path)
df_test_contextual = pd.read_csv(test_contextual)

In [ ]:
df_train.head(2)
display(df_train.head(3))
display(df_train.shape)
display(df_train.label.value_counts())
display(df_train.dataset.value_counts())

print(f'Are tweets ids unique? {df_train["anonymized_tweet_id"].is_unique}')
print(f'Are there any repeated lines? {df_train.duplicated().any()}')


,anonymized_tweet_id,anonymized_text,label,dataset
0,217874450618134,@unknown Con tutte le denunce che si sta becca...,1,training_politics
1,360042217507605,#PRESCRIZIONE : I (t)re magi hanno trovato l’a...,0,training_politics
2,817917484817935,Il #m5s ha votato contro le #unionicivili addu...,1,training_politics


(5600, 4)

,count
label,
0,3456
1,2144


,count
dataset,
training_politics,5600


Are tweets ids unique? True
Are there any repeated lines? False


In [ ]:
print(df_test["dataset"].value_counts())

df_train = df_train.loc[
    df_train["dataset"].eq("training_politics")
].copy()

df_test = df_test.loc[
    df_test["dataset"].eq("test_politics")
].copy()

assert len(df_train) == 5600
assert len(df_test) == 1400

dataset
test_religious    3000
test_politics     1400
Name: count, dtype: int64


In [ ]:
df_test["dataset"].value_counts()

,count
dataset,
test_politics,1400


In [ ]:
df_test.head(2)

,anonymized_tweet_id,anonymized_text,label,dataset
0,424801448454884,questo anno #babbonatale portato giocattolo it...,1,test_politics
1,775194088981616,Adesso che in mezzo alla strada grazie al verg...,1,test_politics


In [ ]:
df_train["anonymized_tweet_id"].is_unique

True

In [ ]:
df_train['anonymized_tweet_id'].duplicated().any()

np.False_

In [ ]:
df_train_contextual.head(2)
display(df_train_contextual.head(3))
display(df_train_contextual.shape)
display(df_train_contextual.columns)
display(df_train.dataset.value_counts())
assert (df_train.anonymized_tweet_id == df_train_contextual.anonymized_tweet_id).all()

print(f'Are tweets ids unique? {df_train_contextual["anonymized_tweet_id"].is_unique}')
print(f'Are there any repeated lines? {df_train_contextual.duplicated().any()}')

,anonymized_tweet_id,created_at,retweet_count,favorite_count,source,is_reply,is_retweet,is_quote,anonymized_user_id,user_created_at,statuses_count,followers_count,friends_count,anonymized_description,dataset
0,217874450618134,2018-08-11,0.0,6.0,"<a href=""http://twitter.com/download/iphone"" r...",1.0,0.0,0.0,8.868893e+14,2018-04-01,554.0,748.0,753.0,Avete presente quegli stereotipi sui siciliani...,training_politics
1,360042217507605,2018-11-08,154.0,448.0,"<a href=""http://twitter.com/download/iphone"" r...",0.0,0.0,0.0,2.570741e+14,2011-10-26,35043.0,10838.0,4535.0,Prof di latino e Deputata PD.Ama lo sport la R...,training_politics
2,817917484817935,2018-11-10,195.0,638.0,"<a href=""http://twitter.com/download/iphone"" r...",0.0,0.0,0.0,6.908473e+14,2009-09-24,139750.0,7971.0,1866.0,"Medico, appassionato di Cinema d'Autore, Music...",training_politics


(5600, 15)

Index(['anonymized_tweet_id', 'created_at', 'retweet_count', 'favorite_count',
       'source', 'is_reply', 'is_retweet', 'is_quote', 'anonymized_user_id',
       'user_created_at', 'statuses_count', 'followers_count', 'friends_count',
       'anonymized_description', 'dataset'],
      dtype='object')

,count
dataset,
training_politics,5600


Are tweets ids unique? True
Are there any repeated lines? False


In [ ]:

politicians_df = pd.read_csv(
    "/content/drive/MyDrive/master_unipa/politicians.csv",
    dtype="string",
    encoding="utf-8-sig",
)

parties_df = pd.read_csv(
    "/content/drive/MyDrive/master_unipa/parties.csv",
    dtype="string",
    encoding="utf-8-sig",
)

In [ ]:
url_count = df_train['anonymized_text'].str.count(r"\[URL\]")

#print("Total [URL] :", df_train['url_count'].sum())
url_count.sum()

np.int64(1585)

Eliminate double white spaces

In [ ]:
df_train['text_clean'] = df_train['anonymized_text'].str.replace(r"\s+", " ", regex=True).str.strip()
#dataset['text_clean'] = dataset['text_clean'].apply(lambda x: " ".join(x.split()))
columns = ['text_clean','label']

df_clean = df_train[columns]
display(df_clean.head(2))
df_clean.to_csv('df_train.csv', index=False)

,text_clean,label
0,@unknown Con tutte le denunce che si sta becca...,1
1,#PRESCRIZIONE : I (t)re magi hanno trovato l’a...,0


### Wiki data : function for importing the wiki data dictionary with names from italian political parties and politicians

In [ ]:
def compact_key(text):
    """ cercare una voce dentro un hashtag."""
    text = unicodedata.normalize("NFC", text).casefold()
    return re.sub(r"[\s_-]+", "", text)


def collect_entries(frame, columns):
    entries = {}

    for column in columns:
        if column not in frame.columns:
            continue

        for value in frame[column].dropna():
            value = str(value).strip()

            # Esclude valori vuoti e identificatori senza etichetta.
            if not value or re.fullmatch(r"Q\d+", value):
                continue

            key = compact_key(value)
            entries.setdefault(key, set()).add(value)

    return entries


politician_entries = collect_entries(
    politicians_df,
    ["nome", "cognome", "alias"],
)

party_entries = collect_entries(
    parties_df,
    ["partitoLabel", "alias", "sigla"],
)

political_entries = {}

for source in [politician_entries, party_entries]:
    for key, variants in source.items():
        political_entries.setdefault(key, set()).update(variants)



## Clean_text

The `clean_text()` function only normalizes whitespace and creates the `text_clean` column, which is used later to identify duplicates prior to the split. `load_split()` then loads this column as `text`. The full preprocessing pipeline is applied subsequently by `preprocess_text()`, before tokenization: it removes URLs and mentions, handles emojis, and annotates political entities within hashtags using PER/ORG markers.

In [ ]:
def clean_text(text):
  return re.sub(r"\s+", " ", text).strip()


df_train['text_clean'] = df_train['anonymized_text'].str.replace(r"\s+", " ", regex=True).str.strip()


In [ ]:
# Load and validate each split
def load_split(path):
    df = pd.read_csv(path)

    text_column = (
        "text_clean" if "text_clean" in df.columns
        else "anonymized_text"
    )

    required = {text_column, "label"}
    if not required.issubset(df.columns):
        raise ValueError(
            f"{path}: expected text_clean or anonymized_text, plus label."
        )

    if df[[text_column, "label"]].isna().any().any():
        raise ValueError(f"{path}: missing text or labels; inspect these rows.")

    labels = pd.to_numeric(df["label"], errors="raise")
    if not labels.isin([0, 1]).all():
        raise ValueError(f"{path}: labels must be 0 or 1.")

    # Already-cleaned input is used unchanged.
    if text_column == "text_clean":
        text = df[text_column].astype(str).str.strip()
    else:
        text = df[text_column].map(clean_text)

    if text.eq("").any():
        raise ValueError(f"{path}: empty text after preprocessing.")

    result = pd.DataFrame({
        "text": text,
        "labels": labels.astype(int),
    })

    print(f"\n{path}: {len(result)} rows")
    print(result["labels"].value_counts().sort_index())

    return result.reset_index(drop=True)

In [ ]:
for df in (df_train, df_test):
    if df["anonymized_text"].isna().any():
        raise ValueError("Missing tweet text: inspect these rows first.")

    df["text_clean"] = df["anonymized_text"].map(clean_text)

display(df_train.head(2))
display(df_test.head(2))
df_test.to_csv('test.csv')

,anonymized_tweet_id,anonymized_text,label,dataset,text_clean
0,217874450618134,@unknown Con tutte le denunce che si sta becca...,1,training_politics,@unknown Con tutte le denunce che si sta becca...
1,360042217507605,#PRESCRIZIONE : I (t)re magi hanno trovato l’a...,0,training_politics,#PRESCRIZIONE : I (t)re magi hanno trovato l’a...


,anonymized_tweet_id,anonymized_text,label,dataset,text_clean
0,424801448454884,questo anno #babbonatale portato giocattolo it...,1,test_politics,questo anno #babbonatale portato giocattolo it...
1,775194088981616,Adesso che in mezzo alla strada grazie al verg...,1,test_politics,Adesso che in mezzo alla strada grazie al verg...


In [ ]:
from sklearn.model_selection import train_test_split

df = df_train.copy()

# Check duplicates before splitting
if df.groupby("text_clean")["label"].nunique().gt(1).any():
    raise ValueError("Some identical texts have conflicting labels.")

df = df.drop_duplicates(subset="text_clean").reset_index(drop=True)

train_df, val_df = train_test_split(
    df,
    test_size=0.10,
    random_state=42,
    stratify=df["label"],  # Preserve class proportions
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

train_df.to_csv("train.csv", index=False)
val_df.to_csv("validation.csv", index=False)

print(f"Train: {len(train_df)} | Validation: {len(val_df)}")
print("Train proportions:\n", train_df["label"].value_counts(normalize=True))
print("Validation proportions:\n", val_df["label"].value_counts(normalize=True))

Train: 5039 | Validation: 560
Train proportions:
 label
0    0.617186
1    0.382814
Name: proportion, dtype: float64
Validation proportions:
 label
0    0.617857
1    0.382143
Name: proportion, dtype: float64


In [ ]:
train_df = load_split("train.csv")
val_df = load_split("validation.csv")
test_df = load_split('test.csv')



train.csv: 5039 rows
labels
0    3110
1    1929
Name: count, dtype: int64

validation.csv: 560 rows
labels
0    346
1    214
Name: count, dtype: int64

test.csv: 1400 rows
labels
0    700
1    700
Name: count, dtype: int64


In [ ]:
def find_overlaps(
    df_a, df_b, name_a, name_b,
    text_col="text", label_col="labels"
):
    def prepare(df, name):
        required = [text_col, label_col]
        missing = [col for col in required if col not in df.columns]

        if missing:
            raise ValueError(
                f"{name}: missing {missing}. "
                f"Available columns: {df.columns.tolist()}"
            )

        result = df[required].copy()
        result[f"row_{name}"] = df.index

        return (
            result
            .dropna(subset=[text_col])
            .rename(columns={label_col: f"label_{name}"})
        )

    matches = prepare(df_a, name_a).merge(
        prepare(df_b, name_b),
        on=text_col,
        how="inner",
    )

    matches["conflicting_labels"] = (
        matches[f"label_{name_a}"] != matches[f"label_{name_b}"]
    )

    print(
        f"{name_a} ↔ {name_b}: "
        f"{matches[text_col].nunique()} distinct shared texts, "
        f"{len(matches)} matching row pairs"
    )

    return matches

In [ ]:
train_val_overlap = find_overlaps(
    train_df, val_df, "train", "validation"
)

train_test_overlap = find_overlaps(
    train_df, test_df, "train", "test"
)

val_test_overlap = find_overlaps(
    val_df, test_df, "validation", "test"
)

display(train_val_overlap)
display(train_test_overlap)
display(val_test_overlap)

train ↔ validation: 0 distinct shared texts, 0 matching row pairs
train ↔ test: 1 distinct shared texts, 1 matching row pairs
validation ↔ test: 0 distinct shared texts, 0 matching row pairs


,text,label_train,row_train,label_validation,row_validation,conflicting_labels


,text,label_train,row_train,label_test,row_test,conflicting_labels
0,@unknown's account is temporarily unavailable ...,1,606,0,419,True


,text,label_validation,row_validation,label_test,row_test,conflicting_labels


Pre processing

In [ ]:
print("Politician columns:", politicians_df.columns.tolist())
print("Party columns:", parties_df.columns.tolist())

print("Politician entries:", len(politician_entries))
print("Party entries:", len(party_entries))

assert politician_entries, "Politician dictionary is empty."
assert party_entries, "Party dictionary is empty."

for key in ["salvini", "meloni", "m5s", "pd"]:
    print(key, "→", political_entries.get(key, "NOT FOUND"))

Politician columns: ['persona', 'nome', 'cognome']
Party columns: ['partito', 'partitoLabel', 'alias', 'sigla']
Politician entries: 4923
Party entries: 1120
salvini → {'Salvini'}
meloni → {'Meloni'}
m5s → {'M5S'}
pd → {'PD'}


In [ ]:
import re
import regex
import emoji


class CustomPreprocessor:
    pattern = re.compile(
        r"(?P<url>(?:https?://|www\.)[^\s<>]+)"
        r"|(?P<user>(?<![\w@])@[A-Za-z0-9_]+)"
        r"|(?P<hashtag>(?<![\w#])#(?:(?!ー)\w)+)",
        flags=re.IGNORECASE,
    )

    COVID_PATTERN = re.compile(
        r"\bcovid(?P<suffix>[\s_\-‐‑‒–—―−﹘﹣－＿ー]*19)?\b",
        flags=re.IGNORECASE,
    )

    def preprocess(self, text):
        if not isinstance(text, str):
            raise TypeError("Fill missing text before preprocessing.")

        def replace(match):
            if match.lastgroup == "url":
                url = match.group()
                trimmed = url.rstrip(".,!?;:")
                return " " + url[len(trimmed):]

            if match.lastgroup == "user":
                return " "

            return f"<hashtag> {match.group()[1:]} </hashtag>"

        # Remove existing URL and [user]
        text = re.sub(
            r"\[url\]|\[user\]|<url>|<user>",
            " ",
            text,
            flags=re.IGNORECASE,
        )

        text = text.translate(str.maketrans({
            "“": '"',
            "”": '"',
            "‘": "'",
            "’": "'",
        }))

        text = self.pattern.sub(replace, text)

        text = self.COVID_PATTERN.sub(
            lambda match: "covid" + (match.group("suffix") or ""),
            text,
        )

        text = re.sub(
            r"([_\-‐‑‒–—―−﹘﹣－＿ー])",
            r" \1 ",
            text,
        )

        replacements = {
            "❤️": " cuore ",
            "❤": " cuore ",
            ":)": " sorriso ",
            ":(": " tristezza ",
        }

        for symbol, replacement in replacements.items():
            text = text.replace(symbol, replacement)

        text = emoji.replace_emoji(
            text,
            replace=lambda chars, data: f" {chars} ",
        )

        return re.sub(r"\s+", " ", text).strip()


prepare_text = CustomPreprocessor()

# Model and tokenizer


In [ ]:
MODEL_NAME = (
    "m-polignano-uniba/"
    "bert_uncased_L-12_H-768_A-12_italian_alb3rt0"
)

SEED = 42
MAX_LENGTH = 256  # Tokens, not characters
OUTPUT_DIR = Path("outputs/alberto_clean_wikidata") # output directory


set_seed(SEED)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    classifier_dropout=0.2,
    id2label={0: "NON_HATE", 1: "HATE"},
    label2id={"NON_HATE": 0, "HATE": 1},
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: m-polignano-uniba/bert_uncased_L-12_H-768_A-12_italian_alb3rt0
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialize

In [ ]:
emoji_counts = Counter()

text_columns = ["text"]
if "description" in train_df.columns:
    text_columns.append("description")

for column in text_columns:
    for text in train_df[column].fillna(""):
        for match in emoji.emoji_list(text):
            emoji_counts[match["emoji"]] += 1

In [ ]:
MIN_EMOJI_COUNT = 2

def unsupported(symbol):
    ids = tokenizer.encode(symbol, add_special_tokens=False)
    return not ids or tokenizer.unk_token_id in ids

emoji_tokens = [
    symbol for symbol, count in emoji_counts.items()
    if count >= MIN_EMOJI_COUNT and unsupported(symbol)
]


training_texts = train_df["text"].fillna("")

# added tokens: each annotation becomes one token
markers = ['<per>', '</per>', '<org>', '</org>', '<hashtag>', '</hashtag>']


other_tokens = ['covid']

candidates = list(dict.fromkeys(emoji_tokens + other_tokens + markers ))


num_added = tokenizer.add_tokens([
    AddedToken(
        token,
        normalized=(token in other_tokens),
        single_word=(token in other_tokens),
        lstrip=False,
        rstrip=False,
    )
    for token in candidates
])


if num_added:
    model.resize_token_embeddings(len(tokenizer))

print("New vocabulary entries:", num_added)
print("Emoji candidates:", emoji_tokens)

for token in candidates:
    assert tokenizer.tokenize(token) == [token], (token, tokenizer.tokenize(token))
assert model.get_input_embeddings().num_embeddings == len(tokenizer)


New vocabulary entries: 105
Emoji candidates: ['💩', '🔵', '🔴', '🖕', '🙄', '😅', '‼', '👇', '😡', '🤔', '📲', '🥎', '🌾', '🌹', '🐬', '😎', '🤚', '🖤', '🇮🇹', '🇱🇷', '🏴\u200d☠️', '‼️', '🤮', '⚡', '🤡', '🤫', '😑', '⬇️', '➡️', '🧐', '🔸', '✌️', '🤭', '💤', '🤬', '🍾', '🙂', '🤣', '🤞', '😬', '😆', '👎', '🎪', '🍒', '🪴', '🤢', '🤦\u200d♂️', '😀', '😒', '😷', '❗', '👌', '🔽', '🤷\u200d♂️', '💪', '🍺', '🤦\u200d♀️', '💸', '😳', '👍🏻', '😐', '💔', '📣', '🤐', '👉', '🌶', '👇🏼', '👎🏻', '🔥', '🤞🏻', '🛑', '🖕🏻', '🐲', '🤗', '🇪🇺', '🤷🏻\u200d♂️', '👋', '🔊', '☠️', '🤦🏻\u200d♂️', '▶️', '🌊', '😶', '🗽', '✅', '❌', '🎶', '❓', '🦄', '🤦🏻\u200d♀️', '👏🏻', '🖐', '😲', '👆', '➡', '✔', '✔️', '⚠️']


Entities matching and preprocessing

In [ ]:
SHORT_PARTY_ALIASES = {"m5s", "pd", "fdi", "lega", "leu"}
EXCLUDE_FROM_SUBSTRING = {"aprile", "buffoni", "rossi"}


person_keys = {
    key for key in politician_entries
    if key.isalnum() and len(key) >= 4
}

party_keys = {
    key for key in party_entries
    if key.isalnum()
    and (len(key) >= 4 or key in SHORT_PARTY_ALIASES)
}

ambiguous_keys = person_keys & party_keys

entity_types = {
    key: "per" for key in person_keys - ambiguous_keys
}
entity_types.update({
    key: "org" for key in party_keys - ambiguous_keys
})


substring_keys = sorted(
    (
        key for key in entity_types
        if (len(key) >= 4 or key == "m5s")
        and key not in EXCLUDE_FROM_SUBSTRING
    ),
    key=lambda key: (-len(key), key),
)




if not substring_keys:
    raise ValueError("No substring keys.")

political_pattern = re.compile(
    "|".join(re.escape(key) for key in substring_keys),
    flags=re.IGNORECASE,
)


def separate_and_tag_political_names(text):
    def process_hashtag(match):
        word = match.group(1).strip()


        key = word.casefold()
        if key in entity_types:
            tag = entity_types[key]
            return f"<hashtag> <{tag}> {word} </{tag}> </hashtag>"

        pieces = []
        cursor = 0

        # find eligible names inside the hashtag.
        for entity in political_pattern.finditer(word):
            pieces.append(word[cursor:entity.start()])

            matched_text = entity.group()
            tag = entity_types[matched_text.casefold()]
            pieces.append(f"<{tag}> {matched_text} </{tag}>")

            cursor = entity.end()

        pieces.append(word[cursor:])
        content = " ".join(piece for piece in pieces if piece)

        return f"<hashtag> {content} </hashtag>"

    return re.sub(
        r"<hashtag>\s*([^<>]*?)\s*</hashtag>",
        process_hashtag,
        text,
    )

def split_numbers_outside_entities(text):
    pattern = regex.compile(
        r"<per>.*?</per>(*SKIP)(*F)"
        r"|<org>.*?</org>(*SKIP)(*F)"
        r"|(?<=\p{L})(?=\p{Nd})"
        r"|(?<=\p{Nd})(?=\p{L})",
        flags=regex.DOTALL,
    )
    return pattern.sub(" ", text)


def preprocess_text(text):  # preprocessing and entitities matching
    cleaned = prepare_text.preprocess(text)
    tagged = separate_and_tag_political_names(cleaned)
    return split_numbers_outside_entities(tagged)



In [ ]:
def tokenize(batch):
    return tokenizer(
        [preprocess_text(text) for text in batch["text"]],
        truncation=True,
        max_length=MAX_LENGTH,
    )

### make dataset

In [ ]:
def make_dataset(frame):
    dataset = Dataset.from_pandas(frame, preserve_index=False)
    return dataset.map(
        tokenize,
        batched=True,
        remove_columns=["text"],
    )


train_dataset = make_dataset(train_df)
val_dataset = make_dataset(val_df)
test_dataset = make_dataset(test_df)

collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/5039 [00:00<?, ? examples/s]

Map:   0%|          | 0/560 [00:00<?, ? examples/s]

Map:   0%|          | 0/1400 [00:00<?, ? examples/s]

Examples

In [ ]:
for text in [
    "Ciao @mario!",
    "Leggi [URL] e commenta.",
    "Guarda https://example.com!",
    "#salvinibugiardo @123 [URL]",
]:
    print(repr(text), "→", repr(preprocess_text(text)))

'Ciao @mario!' → 'Ciao !'
'Leggi [URL] e commenta.' → 'Leggi e commenta.'
'Guarda https://example.com!' → 'Guarda !'
'#salvinibugiardo @123 [URL]' → '<hashtag> <per> salvini </per> bugiardo </hashtag>'


## Trainer and evaluation metrics

- best checkpoint is selected using validation macro-F1

In [ ]:
# Evaluation metrics
def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    if isinstance(logits, tuple):
        logits = logits[0]

    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary",
        pos_label=1,
        zero_division=0,
    )

    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(
            labels,
            predictions,
            labels=[0, 1],
            average="macro",
            zero_division=0,
        ),
        "hate_precision": precision,
        "hate_recall": recall,
        "hate_f1": f1,
    }


# Fine-tuning configuration
training_args = TrainingArguments(
    learning_rate=5e-6,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=8,

    weight_decay=0.01,
    optim="adamw_torch",

    # chosen eval strategy for check the modelling before epoch completition since we have a small amount of training data
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",  # best checkpoint selection
    greater_is_better=True,
    save_total_limit=1,  #disk space clean
    logging_steps=50,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=4,
            early_stopping_threshold=0.0,
        )]
)


# Train
trainer.train()


Step,Training Loss,Validation Loss,Accuracy,Macro F1,Hate Precision,Hate Recall,Hate F1
100,0.485443,0.369337,0.835714,0.829609,0.754167,0.845794,0.797357
200,0.310909,0.257875,0.903571,0.896118,0.908163,0.831776,0.868293
300,0.276322,0.231567,0.917857,0.912381,0.907767,0.873832,0.890476
400,0.269012,0.219543,0.930357,0.925785,0.922705,0.892523,0.907363
500,0.221105,0.219720,0.926786,0.922125,0.913876,0.892523,0.903073
600,0.223155,0.221856,0.923214,0.918624,0.901408,0.897196,0.899297
700,0.143767,0.288437,0.923214,0.919322,0.883408,0.920561,0.901602
800,0.257319,0.241051,0.925000,0.919368,0.934343,0.864486,0.898058


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=800, training_loss=0.2894915574789047, metrics={'train_runtime': 54.1132, 'train_samples_per_second': 236.494, 'train_steps_per_second': 14.784, 'total_flos': 536658164064240.0, 'train_loss': 0.2894915574789047, 'epoch': 2.5396825396825395})

## Prediction on test set and evaluation

In [ ]:
# test evaluation
test_output = trainer.predict(test_dataset)
test_predictions = np.argmax(test_output.predictions, axis=-1)

print("\nTest metrics:")
print(test_output.metrics)

print("\nClassification report:")
print(classification_report(
    test_df["labels"],
    test_predictions,
    labels=[0, 1],
    target_names=["NON_HATE", "HATE"],
    digits=4,
    zero_division=0,
))

print("\nConfusion matrix: rows=true labels, columns=predictions")
print(confusion_matrix(
    test_df["labels"],
    test_predictions,
    labels=[0, 1],
))




results = test_df.rename(columns={"labels": "true_label"}).copy()
results["predicted_label"] = test_predictions
results["probability_hate"] = torch.softmax(
    torch.tensor(test_output.predictions),
    dim=-1,
)[:, 1].numpy()


Test metrics:
{'test_loss': 0.2657780349254608, 'test_accuracy': 0.9107142857142857, 'test_macro_f1': 0.9105956424827029, 'test_hate_precision': 0.9429892141756548, 'test_hate_recall': 0.8742857142857143, 'test_hate_f1': 0.9073387694588584, 'test_runtime': 0.5728, 'test_samples_per_second': 2444.076, 'test_steps_per_second': 76.814}

Classification report:
              precision    recall  f1-score   support

    NON_HATE     0.8828    0.9471    0.9139       700
        HATE     0.9430    0.8743    0.9073       700

    accuracy                         0.9107      1400
   macro avg     0.9129    0.9107    0.9106      1400
weighted avg     0.9129    0.9107    0.9106      1400


Confusion matrix: rows=true labels, columns=predictions
[[663  37]
 [ 88 612]]


### Print best checkpoint results

In [ ]:
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation macro-F1:", trainer.state.best_metric)

Best checkpoint: trainer_output/checkpoint-400
Best validation macro-F1: 0.9257847145056222


In [ ]:
print("Classifier:", model.config.classifier_dropout)
print("Hidden:", model.config.hidden_dropout_prob)
print("Attention:", model.config.attention_probs_dropout_prob)

Classifier: 0.2
Hidden: 0.1
Attention: 0.1


## Error analysis

### Save the predictions on val and test sets for doing an error analysis

In [ ]:

ERROR_DIR = Path(
    "/content/drive/MyDrive/master_unipa/alberto_wiki_error_analysis_final"
)
ERROR_DIR.mkdir(parents=True, exist_ok=True)


def save_error_analysis(trainer, dataset, frame, split_name):

    output = trainer.predict(dataset, metric_key_prefix=split_name)

    logits = output.predictions
    if isinstance(logits, tuple):
        logits = logits[0]

    probabilities = torch.softmax(
        torch.tensor(logits), dim=-1
    )[:, 1].numpy()

    assert len(frame) == len(logits)
    np.testing.assert_array_equal(
        frame["labels"].to_numpy(),
        output.label_ids,
    )

    analysis = frame.copy().reset_index(drop=True)
    analysis["prediction"] = logits.argmax(axis=-1)
    analysis["probability_hate"] = probabilities
    analysis["correct"] = analysis["labels"].eq(analysis["prediction"])

    false_negatives = analysis[
        (analysis["labels"] == 1) & (analysis["prediction"] == 0)
    ].sort_values("probability_hate")

    false_positives = analysis[
        (analysis["labels"] == 0) & (analysis["prediction"] == 1)
    ].sort_values("probability_hate", ascending=False)

    for suffix, result in [
        ("predictions", analysis),
        ("false_negatives", false_negatives),
        ("false_positives", false_positives),
    ]:
        result.to_csv(
            ERROR_DIR / f"{split_name}_{suffix}.csv",
            index=False,
            encoding="utf-8-sig",
        )

    print(f"\n{split_name.upper()}")
    print(f"False negatives: {len(false_negatives)}")
    print(f"False positives: {len(false_positives)}")

    with pd.option_context("display.max_colwidth", None):
        print("\nFalse negatives — most confident mistakes first:")
        display(false_negatives.head(20))

        print("\nFalse positives — most confident mistakes first:")
        display(false_positives.head(20))

    return analysis, false_negatives, false_positives


val_analysis, val_fn, val_fp = save_error_analysis(
    trainer, val_dataset, val_df, "validation"
)

test_analysis, test_fn, test_fp = save_error_analysis(
    trainer, test_dataset, test_df, "test"
)

print(f"\nSaved CSV files to: {ERROR_DIR}")


VALIDATION
False negatives: 23
False positives: 16

False negatives — most confident mistakes first:


,text,labels,prediction,probability_hate,correct
325,"In #senato si sta votando ORA una vera e propria SANATORIA dei MISFATTI del #Governo #Conte. Chiedono la conversione in #Legge del #decreto del #25marzo. In parole povere, VOGLIONO PARARSI IL SEDERE SAPENDO D'AVER AGITO SENZA ALCUNA LEGITTIMAZIONE. @610058340653592 #20maggio",1,0,0.011547,False
261,Buone notizie : l'inferno verrà svuotato per fare posto a #sanitari e #politici,1,0,0.014296,False
269,Sarò sempre grato al #m5s. Far sparire #dibattista è un’opera di grande valore civico.,1,0,0.016211,False
440,Sono già arrivati i #renzioti microchippati che se la cantano e se la suonano sul #RecoveryPlan ? L'hanno già scritto grazie #renzi ? #PNRR,1,0,0.017905,False
16,"@629658163154054 Si, avrebbe potuto accadere altrove. Ma la prima regione infettata dell’Italia è la regione Lombardia sotto il governo #salvinivirus. E le altre no. Questo è un fatto. #coronavirusitalIa",1,0,0.018195,False
439,"Sono iscritto al #M5S dal 2010 e voterò NO per l'autorizzazione a procedere, non voglio più vedere nuovamente l'italia in mano al PD, Berlusconi, e altre bande di criminali #Diciotti #Saviano #17Febbraio #ottoemezzo #mezzorainpiu #m5s #BuonaDomenica #nonelarena #StaseraItalia",1,0,0.019597,False
190,"@868843245148057 #renzi, il tuo socio #Gozi avrà stappato una magnum 🥂🍾🥂, tra poco, il primo trombato di #Macron, potrà scaldare uno scranno a #Bruxelles e fare i suoi interessi.",1,0,0.021481,False
413,Da quando la città di Livorno non è piu amministrata da #Nogarin e dal #m5s non si sente più nulla. Tutto funziona bene e nessun pennivendolo attacca il nuovo sindaco. ECCHECASO. #raggi #roma #SeaWacht3 #Olimpiade2026,1,0,0.053934,False
108,"@652473961427961 Su coraggio, e quanto prima diffondete la lista dei partecipanti (anche se sarà lunga e da dividere zona per zona)! Così sapremo quali saranno i locali da frequentare anche dopo che ci saremo liberati dall'imbecille che si spaccia per #ministro! #ioapro #voiaprite #noiveniamo",1,0,0.054424,False
19,In Italia dovremmo avere la stessa tenacia di #ilariacucchi : non mollare di un metro finché #salvini non finirà al gabbio condannato per le vicende del #Russiagate .,1,0,0.066358,False



False positives — most confident mistakes first:


,text,labels,prediction,probability_hate,correct
252,#MafiaNigeriana ohibò TG5 ha scoperto che in Italia c’è la mafia nigeriana....chissà come è arrivata da noi???! Che dice #Letta che predica iussoli e più sbarchi e #Lamorgesevattene che usa la polizia solo contro gli italiani che manifestano pacificamente perché vogliono lavorare,0,1,0.984129,False
144,Uno dei modi di dire o #dialetto della mia splendida #Pavia riportato sul calendario dialetto pavese. Caro #governodeipeggiori sai cosa significa? Prima o poi si andrà alle #urne... li' farete i conti e saranno amari per molti! [URL],0,1,0.984036,False
105,"Salvini che ""screma"" i commenti negativi bloccando...significa che c'è un calo ed inesorabile calo dei consensi... Tutto questo è bellissimo. #salvinicensuraitaliani #salviniblocca #salviniseiallafrutta #salvinivergognati #salvinicheraccontabufale",0,1,0.981457,False
266,"@804831429960458 Quel partito sembra aver recepito la lezione di un buffone del foro... quel tale che svelava a tutti come, per fare #politica, servano ""anima e cacciavite""!",0,1,0.981406,False
233,La sfiga dei dinosauri fu che all'epoca non c'era Renzi a salvarli. #Renzi #renzivergognati #dinosauri #Albano #27agosto #OpenArms #Salvini,0,1,0.954655,False
12,#Conte #sivergogni e aggiungo #ConteDimettiti #contevaiacasa per il bene dell'Italia e degli Italiani.,0,1,0.949514,False
178,"@62930090500202 La previsione di ""NostraVirus"" a giugno 2020 era di riaprire Tutto e senza mascherine insieme ai fascisti eppoi abbiamo avuto la seconda ondata di morti più della prima! Adesso ""Ritenta e sarai più fortunato"" e speriamo che sto sfigato non ci fa becca a terza ondata! #Salvini",0,1,0.932219,False
258,"Quando il padano parla di #aborto e accusa le donne, ha capito che ci vuole la #partecipazione degli uomini per concepire un figlio ?#Salvini #salvinivergognati",0,1,0.930381,False
199,".Anche oggi fermati 50 #clandestini in zona Bagnoli della Rosandra(TS). Un ex poliziotto che vive nei pressi:""MAI visti pattugliamenti nei boschi, passa chiunque""#TGR #FVG I #portichiusi di #Salvini sono INUTILI! #immigrati #RottaBalcanica #salvinibugiardo",0,1,0.918225,False
158,"#Tommasi proclama lo sciopero dei #calciatori E quelli che stanno inseguendo un pallone in #MilanGenoa #parmaspal e #SampVerona cosa sono,idraulici? Imbianchini? Ps. Non usate più il termine #pagliacci perché a breve l'associazione pagliacci potrebbe querelarvi per diffamazione.",0,1,0.912871,False



TEST
False negatives: 88
False positives: 37

False negatives — most confident mistakes first:


,text,labels,prediction,probability_hate,correct
822,@657509535824434 @536559930021830 Vediamo in quanti firmano per mandare te a lavorare dopo 25 anni di nullafacenza? #petizionemeloniacasa #giorgiameloni #petizione #politici,1,0,0.011337,False
257,"Ieri in Senato si discuteva della questione #Ilva, il senatore #salvini era assente in quanto a #Verona per la #FieraCavalli. C'è un padiglione dedicato agli asini.",1,0,0.011482,False
181,#consultazioni #DirezionePD il #pd di #zingaretti propone ai fessi #m5s l'invasione di #ong con migliaia di #migranti #clandestini che torneranno ad incrementare il business dell'accoglienza. Per non parlare dello #iussoli e la sottomissione ai diktat #ue di Bruxelles,1,0,0.011925,False
102,@unknown #pd #sx e #m5s stanno lavorando per la nostra sostituzione etnica con gli africani. Il 26 maggio i sovranisti vinceranno per legittima difesa! #votoLega,1,0,0.013292,False
321,@unknown Contro il #m5stelle c'è una #associazione a #delinquere di stampo giornalistico. cit @185232423305451,1,0,0.014096,False
235,"Da radio Capital. ""Sulla manovra confermato il deficit al 2,2% per il 2020"". A Bruxelles, sui media italiani tutti zitti. Un anno fa hanno frantumato i c*glioni per il 2,04%. Ha letto male lo speaker oppure lo stato si è rimesso a 90°? #deficit #Conte #m5spd #giallorosso",1,0,0.014975,False
533,Sbaglio o la #regolarizzazione dei #migranti è l'esatto contrario del #decretosicurezza votato dagli stessi #pentacretini?,1,0,0.015532,False
855,#Grillo Non vi distraete. Questa settimana si propone: Corso accelerato per vivere senza fare un ca@@zo e ricevere quasi 14.000 € al mese. Il corso si tiene a #montecitorio da certificato docente leghista iscrizioni gratuite. [URL],1,0,0.015854,False
365,@621514259604154 Ma figurati.. #m5s #pd hanno fatto in modo di arrivare a PD al governo già dal #4marzo Ringraziamo il burattinaio #mattarella e i suoi mandanti,1,0,0.016347,False
743,"Covid, Vittorio Sgarbi contro Roberto Speranza: ""Responsabili di disagio e morte, finirete in tribunale"" [URL]",1,0,0.016823,False



False positives — most confident mistakes first:


,text,labels,prediction,probability_hate,correct
938,#LaTtualità #Draghi #GovernoDraghi #SalviniPagliaccio #Salvini #riaperture #DecretoRiaperture #DecretoCovid #Coprifuoco aprire la foto! [URL],0,1,0.990088,False
1073,"Ma si, continuiamo a non curare i malati... #SperanzaVattene [URL]",0,1,0.984476,False
572,"A tutti i rancorosi che rilanciano #salviniCIALTRONE, ma che razza di vita frustrante hanno per vomitare tanto livore, poi sopratutto contro un'esponente di minoranza. Chi è ossessionato dai #49milioni di @943206476637778, si informino dei 81,6 milioni dell'Unità che pagherà.",0,1,0.982587,False
146,@unknown @unknown Dimostrazione che questo.governo usa le cose che succedono e le strumentalizza a suo modo per avere voti #SalviniDimettiti #salvinivergognati #sequestoeunministro,0,1,0.977776,False
508,"#Report #RenzoRosso #Quartarepubblica Mascherine prezzo fissato 0,50 Il Manzoni scrisse' che durante la peste ci fu il prezzo fissato del PANE, ma siccome a quel prezzo nessuno lo vendeva ci fu una rivoluzione e l' assalto ai forni SONO COSI' IGNORANTI questi del #contedimettiti",0,1,0.977368,False
336,"#dincà ha ragione...la gente non è stupida, ma nel capire il gioco delle tre carte del #pdm5s Buffoni! Altroché #salvini vittima! #tagadala7",0,1,0.976276,False
604,"@797998657209770 Cari signori vi segnalo che gli italiani non sono #coglioni. Poteva far dimettere la #riccardi e quell altro come fece il #m5s. Ripeto, non siamo Coglioni",0,1,0.960896,False
715,"Non mi permetto di giudicare il presidente #Mattarella circa il suo silenzio sull’ingiustizia sociale dei vitalizi ripristinati, penso che, se fossimo un popolo serio, scenderemmo in piazza per fare in modo che questa grande schifezza abbia fine #vitalizi #ladri",0,1,0.955605,False
733,"Qualcuno avvisi la #Lega e #Salvini, anche la mia vicina di casa da la colpa di tutto a #Speranza, almeno possono rifare un twit contro di lui. #Lega #Salvinivergognati [URL]",0,1,0.929376,False
300,"La moglie del proprietario del Papeete candidata con Borgonzoni in Emilia Romagna. Il proprietario è già eletto al parlamento europeo. Quanto ci sono costati un inno, due cubiste e qualche mojito! #salvinipromettecose #salvinivergognati",0,1,0.928991,False



Saved CSV files to: /content/drive/MyDrive/master_unipa/alberto_wiki_error_analysis_final
